In [1]:
# """
# CSE 5526 - HW3 Q4: Recurrent Neural Network with Recurrent Hidden Units
# -----------------------------------------------------------------------
# Implement the forward and backward passes for a two-layer RNN:
#     - Hidden layer: 4 sigmoid neurons (fully recurrent)
#     - Output layer: 1 linear neuron

# You must complete:
#     (1) rnn_forward : compute hidden activations and outputs
#     (2) rnn_backward : compute δ_y(t), δ_h(t), and gradients

# DO NOT modify the provided matrices or data.
# """

import numpy as np


def sigmoid(x):
    # """Sigmoid activation function."""
    return 1 / (1 + np.exp(-x))


def dsigmoid(y):
    # """Derivative of sigmoid when given y = sigmoid(z)."""
    return y * (1 - y)
     

def rnn_forward(X, U, V, W, b, c):
    """
    Compute the forward pass for all time steps.

    Args:
        X : np.ndarray, shape (n_input, T)
            Input sequence.
        U, V, W : weight matrices for input→hidden, hidden→output, hidden→hidden.
        b, c : biases for hidden and output layers.

    Returns:
        y : np.ndarray, shape (T,)
            Output values at each time step.
        h : np.ndarray, shape (n_hidden, T+1)
            Hidden activations (include h[:,0]=0).
        z_list : list of pre-activations (for backward pass).
    """
    # TODO: Initialize h and y arrays
    # TODO: Loop over time steps:
    #        - compute hidden pre-activation z_t
    #        - apply sigmoid
    #        - compute output y_t = V @ h_t + c
    #        - store z_t for backward use
    h = np.zeros((U.shape[1], X.shape[1] + 1))  # h[:,0] = 0
    y = np.zeros(X.shape[1])
    z_list = []
    for t in range(X.shape[1]):
        z_t = U.T @ X[:, t] + W @ h[:, t] + b
        h[:, t + 1] = sigmoid(z_t)
        y[t] = (V @ h[:, t + 1])[0] + c[0]
        z_list.append(z_t)
    return y, h, z_list
     

def rnn_backward(X, d, y, h, z_list, U, V, W):
    """
    Compute the backward pass using BPTT.

    You must compute:
        δ_y(t) for each time step (output deltas)
        δ_h(t) for each time step (hidden deltas)
        Gradients for V, U, W, b, and c.

    Args:
        X, d, y, h, z_list : from forward pass
        U, V, W : weight matrices

    Returns:
        grads : dict containing
            'delta_y', 'delta_h', 'grad_V', 'grad_U', 'grad_W', 'grad_b', 'grad_c'
    """
    # TODO: Initialize arrays for δ_y(t) and δ_h(t)
    # TODO: Compute δ_y(t) = y(t) - d(t)
    # TODO: Loop backward through time:
    #        - propagate error from output and future hidden states
    #        - multiply by sigmoid derivative to get δ_h(t)
    # TODO: Accumulate gradients for all weights and biases
    T = X.shape[1]
    n_hidden = U.shape[1]
    delta_y = np.zeros(T)
    delta_h = np.zeros((n_hidden, T + 1))
    grad_V = np.zeros_like(V)
    grad_U = np.zeros_like(U)
    grad_W = np.zeros_like(W)
    grad_b = np.zeros(n_hidden)
    grad_c = np.zeros(1,)
    for t in reversed(range(T)): 
        delta_y[t] = y[t] - d[t] 
        grad_V += delta_y[t] * h[:, t + 1].reshape(1, -1)
        delta_h[:, t + 1] += V.T[:, 0] * delta_y[t]
        dh = delta_h[:, t + 1] * dsigmoid(h[:, t + 1])
        grad_U += np.outer(X[:, t], dh) 
        grad_W += np.outer(dh, h[:, t])
        grad_b += dh
        grad_c += delta_y[t]
    return {
        'delta_y': delta_y,
        'delta_h': delta_h[:, 1:],  # exclude h[:,0]
        'grad_V': grad_V,
        'grad_U': grad_U,
        'grad_W': grad_W,
        'grad_b': grad_b,
        'grad_c': grad_c
    }


     

def main():
    """Provided constants (DO NOT change)."""
    U = np.array([
        [-1.1106, -0.0596, -0.4091, -0.0648],
        [-0.6809, -0.6610, -1.2819,  1.0004],
        [ 0.0142,  0.3060, -0.2849, -0.8013]
    ])
    V = np.array([[1.1086, -0.4641, 0.6646, -0.5301]])
    W = np.array([
        [ 0.0918, -0.9585,  0.8270, -0.2248],
        [ 0.2738, -0.3263, -0.1596, -1.3147],
        [-2.1335,  2.4787, -0.8577, -0.0218],
        [ 0.4040,  1.5391,  0.3984, -0.8801]
    ])
    b = np.array([0.0409, 0.1551, -0.3211, 2.1458])
    c = np.array([0.8004])

    X = np.array([
        [-2.5092,  1.9732, -8.8383,  4.1615,  6.6489],
        [ 9.0143, -6.8796,  7.3235, -9.5883, -5.7532],
        [ 4.6399, -6.8801,  2.0223,  9.3982, -6.3635]
    ])
    d = np.array([1.1963, 2.0975, 0.3877, 1.0453, -1.3718])

    # Run once implemented
    try:
        y, h, z_list = rnn_forward(X, U, V, W, b, c)
        grads = rnn_backward(X, d, y, h, z_list, U, V, W)

        print("Output sequence y(t):", y)
        print("Output deltas δ_y(t):", grads.get("delta_y"))
        print("Hidden deltas δ_h(t):", grads.get("delta_h"))
    except NotImplementedError:
        print("Implement rnn_forward and rnn_backward first.")
    except Exception as e:
        print("Error during execution:", e)


if __name__ == "__main__":
    main()

Output sequence y(t): [0.3055426  1.86937853 1.36542787 1.96701192 0.61149548]
Output deltas δ_y(t): [-0.8907574  -0.22812147  0.97772787  0.92171192  1.98329548]
Hidden deltas δ_h(t): [[-0.98749365 -0.25289546  1.08390912  1.02180983  2.19868137]
 [ 0.41340051  0.10587117 -0.4537635  -0.4277665  -0.92044743]
 [-0.59199737 -0.15160953  0.64979794  0.61256974  1.31809817]
 [ 0.4721905   0.12092719 -0.51829354 -0.48859949 -1.05134493]]
